# ERCOT DAM + Weather + Renewables Trading Notebook

This notebook assembles an ERCOT-wide view of load zone prices, load, weather, wind, and solar generation to support trading and storage strategies.

We proceed in four stages:

1. **Setup & configuration**: imports, environment variables, and API keys.
2. **DAM + load + weather data**: load the pre-joined ERCOT dataset and build region-level views.
3. **Price visualization**: generate an interactive ERCOT LZ settlement price overlay for presentation.
4. **Wind + solar fundamentals**: pull hourly renewables data from GridStatus, shape it into regional buckets, and combine into a clean `renewables` panel for downstream modeling.

## 1. Setup and configuration

Load core libraries, plotting tools, and the GridStatus client. The API key is read securely from `.env` via `GRIDSTATUS_API_KEY`.

In [231]:
from dotenv import load_dotenv
import os
import pandas as pd
from gridstatusio import GridStatusClient
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [232]:
load_dotenv()
api_key = os.getenv("GRIDSTATUS_API_KEY")

In [233]:
data = pd.read_csv('data/All_2020_2025_with_AS.csv')

## 2. Load and clean ERCOT DAM + load + weather panel

We load a pre-joined hourly ERCOT dataset (`All_2020_2025_with_AS.csv`), fix the ERCOT "hour ending 24" timestamp convention, set a proper datetime index, and trim to the competitive load zones plus weather and calendar features used later.

In [234]:
data.columns

Index(['COAST_Load', 'EAST_Load', 'ERCOT_Load', 'FWEST_Load', 'HB_BUSAVG_DAM',
       'HB_BUSAVG_RTM', 'HB_HOUSTON_DAM', 'HB_HOUSTON_RTM', 'HB_HUBAVG_DAM',
       'HB_HUBAVG_RTM', 'HB_NORTH_DAM', 'HB_NORTH_RTM', 'HB_PAN_DAM',
       'HB_PAN_RTM', 'HB_SOUTH_DAM', 'HB_SOUTH_RTM', 'HB_WEST_DAM',
       'HB_WEST_RTM', 'LZ_AEN_DAM', 'LZ_AEN_RTM', 'LZ_CPS_DAM', 'LZ_CPS_RTM',
       'LZ_HOUSTON_DAM', 'LZ_HOUSTON_RTM', 'LZ_LCRA_DAM', 'LZ_LCRA_RTM',
       'LZ_NORTH_DAM', 'LZ_NORTH_RTM', 'LZ_RAYBN_DAM', 'LZ_RAYBN_RTM',
       'LZ_SOUTH_DAM', 'LZ_SOUTH_RTM', 'LZ_WEST_DAM', 'LZ_WEST_RTM',
       'NCENT_Load', 'NORTH_Load', 'SCENT_Load', 'SOUTH_Load', 'WEST_Load',
       'altimeter_C', 'altimeter_E', 'altimeter_N', 'altimeter_S',
       'altimeter_W', 'dew_point_temperature_C', 'dew_point_temperature_E',
       'dew_point_temperature_N', 'dew_point_temperature_S',
       'dew_point_temperature_W', 'holiday', 'key', 'month',
       'relative_humidity_C', 'relative_humidity_E', 'relative_humidity_N'

In [235]:
data['key'] = data['key'].str.replace(' 24', ' 0').pipe(
    lambda s: pd.to_datetime(s, format='%m/%d/%Y %H') + 
              pd.to_timedelta(data['key'].str.endswith(' 24').astype(int), unit='D')
)
data = data.set_index('key')

In [236]:
data = data[data.columns[~data.columns.str.contains('HB', regex=True)]]
data = data[data.columns[~data.columns.str.contains('RTM', regex=True)]]
# Keep only the 4 competitive load zones
competitive_lz = ['LZ_NORTH_DAM', 'LZ_SOUTH_DAM', 'LZ_WEST_DAM', 'LZ_HOUSTON_DAM']
noie_lz = ['LZ_AEN_DAM', 'LZ_CPS_DAM', 'LZ_LCRA_DAM']

data = data.drop(columns=noie_lz)
data.columns
cols_to_drop = [
    # Load zones
    'FWEST_Load', 'NCENT_Load', 'SCENT_Load',
    # Ancillary services
    'REGDN', 'REGUP', 'RRS', 'NSPIN', 'ECRS'
]

data = data.drop(columns=cols_to_drop)

In [237]:
data.columns

Index(['COAST_Load', 'EAST_Load', 'ERCOT_Load', 'LZ_HOUSTON_DAM',
       'LZ_NORTH_DAM', 'LZ_RAYBN_DAM', 'LZ_SOUTH_DAM', 'LZ_WEST_DAM',
       'NORTH_Load', 'SOUTH_Load', 'WEST_Load', 'altimeter_C', 'altimeter_E',
       'altimeter_N', 'altimeter_S', 'altimeter_W', 'dew_point_temperature_C',
       'dew_point_temperature_E', 'dew_point_temperature_N',
       'dew_point_temperature_S', 'dew_point_temperature_W', 'holiday',
       'month', 'relative_humidity_C', 'relative_humidity_E',
       'relative_humidity_N', 'relative_humidity_S', 'relative_humidity_W',
       'temperature_C', 'temperature_E', 'temperature_N', 'temperature_S',
       'temperature_W', 'visibility_C', 'visibility_E', 'visibility_N',
       'visibility_S', 'visibility_W', 'weekend', 'Year'],
      dtype='object')

Mixing and matching. 

Load_COAST,  'LZ_HOUSTON_DAM', 'altimeter_C', 'dew_point_temperature_C', relative_humidity_C, temperature_C, visibility_C

Load_SOUTH, 'LZ_SOUTH_DAM', 'altimeter_S', 'dew_point_temperature_S', relative_humidity_S, temperature_S, visibility_S

Load_WEST, 'LZ_WEST_DAM', 'altimeter_W',  'dew_point_temperature_W', relative_humidity_W, temperature_W, visibility_W

Load_NORTH, 'LZ_NORTH_DAM, 'altimeter_N', 'dew_point_temperature_N', relative_humidity_N, temperature_N, visibility_N

Load_EAST, LZ_RAYBN_DAM, 'altimeter_E', 'dew_point_temperature_E', relative_humidity_E, temperature_E, visibility_E






Altimeter (Pressure):
High pressure → clear skies, temperature extremes, suppressed wind generation → load spikes with thin renewable supply → DAM prices spike, evening ramp steepens, battery charge windows compress.

Low pressure → cloud cover, milder temps, wind ramps up (especially West Texas) → load softens while supply surges → DAM prices suppress, spread between on-peak and off-peak narrows, battery arbitrage value shrinks.

--

Visibility:
High visibility → clear, dry air mass, strong solar irradiance → solar output peaks, but afternoon heat load builds fast → DAM on-peak prices elevated, solar-heavy hours see price suppression then sharp evening ramp as solar drops off — key battery dispatch signal.

Low visibility → fog, haze, or storm systems → solar generation curtailed unexpectedly → supply shortfall risk in morning hours → DAM prices can spike early, arbitrage windows shift earlier than typical, operators need to front-load charge cycles.

--

Humidity:
Low humidity → dry heat, efficient turbine output, load driven purely by temperature → DAM prices follow temp curve predictably, battery scheduling straightforward.

High humidity → heat index amplifies felt temperature beyond dry-bulb readings → AC load surges nonlinearly, gas turbine efficiency degrades simultaneously → DAM prices spike harder and faster than temperature alone suggests, peak windows widen, battery should hold discharge for later — the real price extreme often comes 1–2 hours after the temperature peak.

--

Dew Point:
Low dew point → dry air mass, heat feels manageable → load stays moderate even at high temps, prices behave predictably.

High dew point (above 65°F) → sticky, oppressive air regardless of cloud cover → sustained overnight load as buildings struggle to cool down, no overnight price relief → DAM off-peak prices elevated, battery overnight charge window is shorter and more expensive than usual, traders should expect a flatter daily price curve with less arbitrage spread.

In [238]:
data.columns

Index(['COAST_Load', 'EAST_Load', 'ERCOT_Load', 'LZ_HOUSTON_DAM',
       'LZ_NORTH_DAM', 'LZ_RAYBN_DAM', 'LZ_SOUTH_DAM', 'LZ_WEST_DAM',
       'NORTH_Load', 'SOUTH_Load', 'WEST_Load', 'altimeter_C', 'altimeter_E',
       'altimeter_N', 'altimeter_S', 'altimeter_W', 'dew_point_temperature_C',
       'dew_point_temperature_E', 'dew_point_temperature_N',
       'dew_point_temperature_S', 'dew_point_temperature_W', 'holiday',
       'month', 'relative_humidity_C', 'relative_humidity_E',
       'relative_humidity_N', 'relative_humidity_S', 'relative_humidity_W',
       'temperature_C', 'temperature_E', 'temperature_N', 'temperature_S',
       'temperature_W', 'visibility_C', 'visibility_E', 'visibility_N',
       'visibility_S', 'visibility_W', 'weekend', 'Year'],
      dtype='object')

In [239]:
def create_regional_dfs(data: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Splits the master ERCOT DAM DataFrame into region-specific DataFrames,
    each containing its load, SPP, and weather indicators.

    Parameters
    ----------
    data : pd.DataFrame
        Master DataFrame with datetime index and all ERCOT features.

    Returns
    -------
    dict[str, pd.DataFrame]
        Keys are region names, values are region-specific DataFrames.
    """

    # Shared calendar features included in every regional df
    calendar_cols = ['holiday', 'weekend', 'month', 'Year']

    region_cols = {
        'coast': [
            'COAST_Load', 'LZ_HOUSTON_DAM',
            'altimeter_C', 'dew_point_temperature_C',
            'relative_humidity_C', 'temperature_C', 'visibility_C'
        ],
        'south': [
            'SOUTH_Load', 'LZ_SOUTH_DAM',
            'altimeter_S', 'dew_point_temperature_S',
            'relative_humidity_S', 'temperature_S', 'visibility_S'
        ],
        'west': [
            'WEST_Load', 'LZ_WEST_DAM',
            'altimeter_W', 'dew_point_temperature_W',
            'relative_humidity_W', 'temperature_W', 'visibility_W'
        ],
        'north': [
            'NORTH_Load', 'LZ_NORTH_DAM',
            'altimeter_N', 'dew_point_temperature_N',
            'relative_humidity_N', 'temperature_N', 'visibility_N'
        ],
        'east': [
            'EAST_Load', 'LZ_RAYBN_DAM',
            'altimeter_E', 'dew_point_temperature_E',
            'relative_humidity_E', 'temperature_E', 'visibility_E'
        ],
    }

    regional_dfs = {}

    for region, cols in region_cols.items():
        all_cols = cols + calendar_cols
        missing = [c for c in all_cols if c not in data.columns]
        if missing:
            print(f"[WARNING] {region.upper()}: missing columns {missing} — skipped.")
            continue

        regional_dfs[region] = data[all_cols].copy()

    return regional_dfs

In [240]:
regions = create_regional_dfs(data)

# Access individual regions
df_coast = regions['coast']
df_south = regions['south']
df_west  = regions['west']
df_north = regions['north']
df_east  = regions['east']

In [241]:
df_coast.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 51350 entries, 2020-01-01 01:00:00 to 2025-11-09 00:00:00
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   COAST_Load               50408 non-null  float64
 1   LZ_HOUSTON_DAM           51345 non-null  float64
 2   altimeter_C              40827 non-null  float64
 3   dew_point_temperature_C  40785 non-null  float64
 4   relative_humidity_C      40785 non-null  float64
 5   temperature_C            40790 non-null  float64
 6   visibility_C             39962 non-null  float64
 7   holiday                  51350 non-null  int64  
 8   weekend                  51350 non-null  int64  
 9   month                    51350 non-null  int64  
 10  Year                     51280 non-null  float64
dtypes: float64(8), int64(3)
memory usage: 4.7 MB


In [242]:

print(os.path.abspath('ercot_lz_spp.html'))

/Users/habibdjaroueh/Desktop/Modo/ercot_lz_spp.html


In [243]:


def plot_lz_spp_overlay(data: pd.DataFrame) -> None:
    """
    Plots an overlay of LZ Settlement Point Prices for all ERCOT hubs
    across all years in the dataset. Opens directly in browser.

    Parameters
    ----------
    data : pd.DataFrame
        Master DataFrame with a datetime index and LZ SPP columns.
    """

    lz_config = {
        'LZ_HOUSTON_DAM': {'label': 'Houston (Coast)', 'color': '#38bdf8'},
        'LZ_NORTH_DAM':   {'label': 'North',           'color': '#34d399'},
        'LZ_SOUTH_DAM':   {'label': 'South',           'color': '#fb923c'},
        'LZ_WEST_DAM':    {'label': 'West',            'color': '#a78bfa'},
        'LZ_RAYBN_DAM':   {'label': 'East (Rayburn)',  'color': '#f472b6'},
    }

    lz_cols   = list(lz_config.keys())
    available = [c for c in lz_cols if c in data.columns]
    daily     = data[available].resample('D').mean()
    years     = sorted(data.index.year.unique())

    fig = make_subplots(
        rows=2, cols=1,
        row_heights=[0.65, 0.35],
        vertical_spacing=0.08,
        subplot_titles=(
            'Daily Mean LZ Settlement Point Price — All Hubs',
            'Annual Price Distribution by Hub'
        )
    )

    # ── row 1: time series overlay ─────────────────────────────────────────
    for col in available:
        cfg = lz_config[col]
        fig.add_trace(
            go.Scatter(
                x=daily.index,
                y=daily[col],
                name=cfg['label'],
                line=dict(color=cfg['color'], width=1.2),
                opacity=0.85,
                hovertemplate=(
                    f"<b>{cfg['label']}</b><br>"
                    "%{x|%b %d, %Y}<br>"
                    "Price: $%{y:.2f}/MWh<extra></extra>"
                )
            ),
            row=1, col=1
        )

    # ── row 2: annual box plots ────────────────────────────────────────────
    for col in available:
        cfg = lz_config[col]
        for year in years:
            yr_data = data.loc[data.index.year == year, col].dropna()
            fig.add_trace(
                go.Box(
                    x=[str(year)] * len(yr_data),
                    y=yr_data,
                    name=cfg['label'],
                    marker_color=cfg['color'],
                    line=dict(color=cfg['color']),
                    opacity=0.7,
                    showlegend=False,
                    boxmean='sd',
                    hovertemplate=(
                        f"<b>{cfg['label']} {year}</b><br>"
                        "Median: $%{median:.2f}<br>"
                        "Q1: $%{q1:.2f} | Q3: $%{q3:.2f}<extra></extra>"
                    )
                ),
                row=2, col=1
            )

    # ── layout ─────────────────────────────────────────────────────────────
    fig.update_layout(
        height=900,
        template='plotly_dark',
        paper_bgcolor='#0a0e17',
        plot_bgcolor='#111827',
        font=dict(family='monospace', color='#94a3b8', size=11),
        title=dict(
            text='<b>ERCOT DAM — LZ Settlement Point Prices</b>',
            font=dict(size=18, color='#e2e8f0'),
            x=0.01
        ),
        legend=dict(
            orientation='h',
            yanchor='bottom', y=1.01,
            xanchor='left',   x=0,
            bgcolor='rgba(0,0,0,0)',
        ),
        hovermode='x unified',
        margin=dict(l=60, r=40, t=80, b=60),
    )

    axis_style = dict(gridcolor='#1e2d40', zerolinecolor='#1e2d40', linecolor='#1e2d40')
    fig.update_xaxes(**axis_style)
    fig.update_yaxes(**axis_style)
    fig.update_yaxes(title_text='$/MWh', row=1, col=1)
    fig.update_yaxes(title_text='$/MWh', row=2, col=1)
    fig.update_xaxes(title_text='Year',  row=2, col=1)

    fig.update_xaxes(
        rangeselector=dict(
            buttons=[
                dict(count=3,  label='3M',  step='month', stepmode='backward'),
                dict(count=6,  label='6M',  step='month', stepmode='backward'),
                dict(count=1,  label='1Y',  step='year',  stepmode='backward'),
                dict(count=2,  label='2Y',  step='year',  stepmode='backward'),
                dict(step='all', label='ALL')
            ],
            bgcolor='#111827',
            activecolor='#1e3a5f',
            bordercolor='#1e2d40',
            font=dict(color='#94a3b8', size=10),
        ),
        row=1, col=1
    )

    # ── write and open in browser automatically ────────────────────────────
    output_path = 'ercot_lz_spp.html'
    fig.write_html(output_path, auto_open=True)
    print(f"✅ Chart saved and opened: {output_path}")


# ── usage ──────────────────────────────────────────────────────────────────
plot_lz_spp_overlay(data)

✅ Chart saved and opened: ercot_lz_spp.html


In [244]:
data.index

DatetimeIndex(['2020-01-01 01:00:00', '2020-01-01 02:00:00',
               '2020-01-01 03:00:00', '2020-01-01 04:00:00',
               '2020-01-01 05:00:00', '2020-01-01 06:00:00',
               '2020-01-01 07:00:00', '2020-01-01 08:00:00',
               '2020-01-01 09:00:00', '2020-01-01 10:00:00',
               ...
               '2025-11-08 15:00:00', '2025-11-08 16:00:00',
               '2025-11-08 17:00:00', '2025-11-08 18:00:00',
               '2025-11-08 19:00:00', '2025-11-08 20:00:00',
               '2025-11-08 21:00:00', '2025-11-08 22:00:00',
               '2025-11-08 23:00:00', '2025-11-09 00:00:00'],
              dtype='datetime64[ns]', name='key', length=51350, freq=None)

In [245]:
client = GridStatusClient(api_key)
# Fetch data as pandas DataFrame
df = client.get_dataset(
  dataset="ercot_wind_actual_and_forecast_by_geo_region_hourly",
  start="2020-01-01",
  end="2025-11-09",
  publish_time="latest",
  timezone="market",
)

2026-02-26 12:32:07 - INFO - Fetching Page 1...
2026-02-26 12:32:07 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_wind_actual_and_forecast_by_geo_region_hourly/query
2026-02-26 12:32:07 - INFO - Params: {'start_time': Timestamp('2020-01-01 00:00:00'), 'end_time': Timestamp('2025-11-09 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': 'latest', 'timezone': 'market', 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-02-26 12:32:27 - INFO - Done in 20.18 seconds. 
2026-02-26 12:32:27 - INFO - Fetching Page 2...
2026-02-26 12:32:27 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_wind_actual_and_forecast_by_geo_region_hourly/query
2026-02-26 12:32:27 - INFO - Params: {'start_time': Timestamp('2020-01-01 00:00:00'), 'end_time'

In [246]:
df.columns

Index(['interval_start_local', 'interval_start_utc', 'interval_end_local',
       'interval_end_utc', 'publish_time_local', 'publish_time_utc',
       'gen_system_wide', 'cop_hsl_system_wide', 'stwpf_system_wide',
       'wgrpp_system_wide', 'hsl_system_wide', 'gen_panhandle',
       'cop_hsl_panhandle', 'stwpf_panhandle', 'wgrpp_panhandle',
       'gen_coastal', 'cop_hsl_coastal', 'stwpf_coastal', 'wgrpp_coastal',
       'gen_south', 'cop_hsl_south', 'stwpf_south', 'wgrpp_south', 'gen_west',
       'cop_hsl_west', 'stwpf_west', 'wgrpp_west', 'gen_north',
       'cop_hsl_north', 'stwpf_north', 'wgrpp_north'],
      dtype='object')

In [247]:


# 1) Parse timestamps (local)
df["interval_start_local"] = pd.to_datetime(df["interval_start_local"])
df["interval_end_local"]   = pd.to_datetime(df["interval_end_local"])

# Optional: make them naive so display is 'YYYY-MM-DD HH:MM:SS'
for c in ["interval_start_local", "interval_end_local"]:
    if getattr(df[c].dt, "tz", None) is not None:
        df[c] = df[c].dt.tz_convert(None)

# 2) Hourly datetime label (use interval start)
df["datetime"] = df["interval_start_local"].dt.floor("H")

# (optional) sanity check each row is 1 hour
bad = (df["interval_end_local"] - df["interval_start_local"]) != pd.Timedelta(hours=1)
if bad.any():
    print("Warning: some intervals are not 1 hour. Example rows:", df.index[bad].tolist()[:10])

# 3) Wind generation (MW): total + regions that actually exist in your df
wind_gen = (
    df[[
        "datetime",
        "gen_system_wide",
        "gen_west",
        "gen_north",
        "gen_south",
        "gen_coastal",
        "gen_panhandle",
    ]]
    .rename(columns={
        "gen_system_wide": "wind_gen_total_mw",
        "gen_west": "wind_gen_west_mw",
        "gen_north": "wind_gen_north_mw",
        "gen_south": "wind_gen_south_mw",
        "gen_coastal": "wind_gen_coast_mw",
        "gen_panhandle": "wind_gen_panhandle_mw",
    })
    .sort_values("datetime")
    .reset_index(drop=True)
)

wind_gen

/var/folders/dn/6223lsgx1rb1hc1bkl9c51_h0000gn/T/ipykernel_80367/2641652634.py:11: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



,datetime,wind_gen_total_mw,wind_gen_west_mw,wind_gen_north_mw,wind_gen_south_mw,wind_gen_coast_mw,wind_gen_panhandle_mw
0,2020-01-01 06:00:00,12841.91,7041.21,877.15,775.55,455.17,3692.83
1,2020-01-01 07:00:00,13339.68,7762.20,863.53,597.16,441.87,3674.92
2,2020-01-01 08:00:00,13925.54,8339.81,814.34,552.73,553.51,3665.15
3,2020-01-01 09:00:00,14457.71,8976.25,795.03,480.88,536.65,3668.90
4,2020-01-01 10:00:00,14735.06,9267.10,796.84,612.93,384.84,3673.35
...,...,...,...,...,...,...,...
51325,2025-11-09 01:00:00,22599.33,15676.89,1303.97,1030.21,1965.94,2622.32
51326,2025-11-09 02:00:00,23877.53,15935.27,1701.29,1732.08,1966.53,2542.36
51327,2025-11-09 03:00:00,24177.94,16078.67,1860.77,1820.09,1941.02,2477.39
51328,2025-11-09 04:00:00,23708.52,15500.13,2085.25,1775.57,1771.19,2576.38


In [248]:

# Recommended: set GRIDSTATUS_API_KEY as an
# environment variable instead of hardcoding
client = GridStatusClient(api_key)
# Fetch data as pandas DataFrame
df_solar = client.get_dataset(
  dataset="ercot_solar_actual_and_forecast_by_geo_region_hourly",
  start="2020-01-01",
  end="2025-11-09",
  publish_time="latest",
  timezone="market",
)

2026-02-26 12:32:33 - INFO - Fetching Page 1...
2026-02-26 12:32:33 - INFO - GET https://api.gridstatus.io/v1/datasets/ercot_solar_actual_and_forecast_by_geo_region_hourly/query
2026-02-26 12:32:33 - INFO - Params: {'start_time': Timestamp('2020-01-01 00:00:00'), 'end_time': Timestamp('2025-11-09 00:00:00'), 'publish_time_start': None, 'publish_time_end': None, 'limit': None, 'page': 1, 'page_size': None, 'resample_frequency': None, 'resample_by': None, 'resample_function': None, 'publish_time': 'latest', 'timezone': 'market', 'cursor': '', 'filter_column': None, 'filter_value': None, 'filter_operator': '=', 'return_format': 'json', 'json_schema': 'array-of-arrays'}
2026-02-26 12:32:46 - INFO - Done in 13.89 seconds. 
2026-02-26 12:32:46 - INFO - Total number of rows: 29501


In [249]:
df_solar.columns

Index(['interval_start_local', 'interval_start_utc', 'interval_end_local',
       'interval_end_utc', 'publish_time_local', 'publish_time_utc',
       'gen_system_wide', 'cop_hsl_system_wide', 'stppf_system_wide',
       'pvgrpp_system_wide', 'hsl_system_wide', 'gen_centerwest',
       'cop_hsl_centerwest', 'stppf_centerwest', 'pvgrpp_centerwest',
       'gen_northwest', 'cop_hsl_northwest', 'stppf_northwest',
       'pvgrpp_northwest', 'gen_farwest', 'cop_hsl_farwest', 'stppf_farwest',
       'pvgrpp_farwest', 'gen_fareast', 'cop_hsl_fareast', 'stppf_fareast',
       'pvgrpp_fareast', 'gen_southeast', 'cop_hsl_southeast',
       'stppf_southeast', 'pvgrpp_southeast', 'gen_centereast',
       'cop_hsl_centereast', 'stppf_centereast', 'pvgrpp_centereast'],
      dtype='object')

In [250]:


# 1) Parse timestamps (local)
df_solar["interval_start_local"] = pd.to_datetime(df_solar["interval_start_local"])
df_solar["interval_end_local"]   = pd.to_datetime(df_solar["interval_end_local"])

# Optional: make timestamps naive so display is 'YYYY-MM-DD HH:MM:SS'
for c in ["interval_start_local", "interval_end_local"]:
    if getattr(df_solar[c].dt, "tz", None) is not None:
        df_solar[c] = df_solar[c].dt.tz_convert(None)

# 2) Hourly datetime label (use interval start)
df_solar["datetime"] = df_solar["interval_start_local"].dt.floor("H")

# Optional: sanity check 1-hour intervals
bad = (df_solar["interval_end_local"] - df_solar["interval_start_local"]) != pd.Timedelta(hours=1)
if bad.any():
    print("Warning: some intervals are not 1 hour. Example rows:", df_solar.index[bad].tolist()[:10])

# 3) Build "same shape" solar generation output (MW)
#    Solar actual generation columns are gen_* in MW
solar_gen = pd.DataFrame({
    "datetime": df_solar["datetime"],
    "solar_gen_total_mw": df_solar["gen_system_wide"],

    # Aggregations to match your wind-style 5 buckets
    "solar_gen_west_mw": (
        df_solar["gen_centerwest"]
        + df_solar["gen_northwest"]
        + df_solar["gen_farwest"]
    ),
    "solar_gen_north_mw": df_solar["gen_northwest"],
    "solar_gen_east_mw": (df_solar["gen_centereast"] + df_solar["gen_fareast"]),
    "solar_gen_south_mw": df_solar["gen_southeast"],

    # No true "coastal" in this solar schema; using southeast as closest proxy.
    "solar_gen_coast_mw": df_solar["gen_southeast"],
}).sort_values("datetime").reset_index(drop=True)

solar_gen

/var/folders/dn/6223lsgx1rb1hc1bkl9c51_h0000gn/T/ipykernel_80367/3115910155.py:11: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.



,datetime,solar_gen_total_mw,solar_gen_west_mw,solar_gen_north_mw,solar_gen_east_mw,solar_gen_south_mw,solar_gen_coast_mw
0,2022-06-28 21:00:00,7622.22,4981.33,312.51,2301.29,339.60,339.60
1,2022-06-28 22:00:00,6158.26,4228.09,272.31,1739.10,191.07,191.07
2,2022-06-28 23:00:00,4912.19,3383.66,243.89,1367.52,161.01,161.01
3,2022-06-29 00:00:00,2498.09,1845.91,131.67,595.73,56.45,56.45
4,2022-06-29 01:00:00,226.75,186.58,17.66,34.39,5.78,5.78
...,...,...,...,...,...,...,...
29496,2025-11-09 01:00:00,0.94,0.25,0.00,0.69,0.00,0.00
29497,2025-11-09 02:00:00,0.90,0.16,0.00,0.74,0.00,0.00
29498,2025-11-09 03:00:00,0.95,0.14,0.00,0.81,0.00,0.00
29499,2025-11-09 04:00:00,0.77,0.13,0.00,0.64,0.00,0.00


In [251]:
renewables = (
    wind_gen
        .merge(solar_gen, on="datetime", how="inner")
        .sort_values("datetime")
        .set_index("datetime")
)

renewables

,wind_gen_total_mw,wind_gen_west_mw,wind_gen_north_mw,wind_gen_south_mw,wind_gen_coast_mw,wind_gen_panhandle_mw,solar_gen_total_mw,solar_gen_west_mw,solar_gen_north_mw,solar_gen_east_mw,solar_gen_south_mw,solar_gen_coast_mw
datetime,,,,,,,,,,,,
2022-06-28 21:00:00,6542.80,3579.20,521.56,281.26,1876.03,284.75,7622.22,4981.33,312.51,2301.29,339.60,339.60
2022-06-28 22:00:00,8378.65,4282.12,458.95,830.66,2423.85,383.07,6158.26,4228.09,272.31,1739.10,191.07,191.07
2022-06-28 23:00:00,9507.46,4489.07,315.71,1876.86,2342.38,483.44,4912.19,3383.66,243.89,1367.52,161.01,161.01
2022-06-29 00:00:00,9951.63,5109.52,197.59,1863.15,1926.23,855.14,2498.09,1845.91,131.67,595.73,56.45,56.45
2022-06-29 01:00:00,9342.69,4835.75,172.80,1562.58,1320.78,1450.78,226.75,186.58,17.66,34.39,5.78,5.78
...,...,...,...,...,...,...,...,...,...,...,...,...
2025-11-09 01:00:00,22599.33,15676.89,1303.97,1030.21,1965.94,2622.32,0.94,0.25,0.00,0.69,0.00,0.00
2025-11-09 02:00:00,23877.53,15935.27,1701.29,1732.08,1966.53,2542.36,0.90,0.16,0.00,0.74,0.00,0.00
2025-11-09 03:00:00,24177.94,16078.67,1860.77,1820.09,1941.02,2477.39,0.95,0.14,0.00,0.81,0.00,0.00


In [252]:
# Join renewables with DAM + load + weather panel on the datetime index
er = (
    renewables
        .join(data, how="inner")  # index-on-index join (both indexed by datetime)
        .sort_index()
)

er.columns

Index(['wind_gen_total_mw', 'wind_gen_west_mw', 'wind_gen_north_mw',
       'wind_gen_south_mw', 'wind_gen_coast_mw', 'wind_gen_panhandle_mw',
       'solar_gen_total_mw', 'solar_gen_west_mw', 'solar_gen_north_mw',
       'solar_gen_east_mw', 'solar_gen_south_mw', 'solar_gen_coast_mw',
       'COAST_Load', 'EAST_Load', 'ERCOT_Load', 'LZ_HOUSTON_DAM',
       'LZ_NORTH_DAM', 'LZ_RAYBN_DAM', 'LZ_SOUTH_DAM', 'LZ_WEST_DAM',
       'NORTH_Load', 'SOUTH_Load', 'WEST_Load', 'altimeter_C', 'altimeter_E',
       'altimeter_N', 'altimeter_S', 'altimeter_W', 'dew_point_temperature_C',
       'dew_point_temperature_E', 'dew_point_temperature_N',
       'dew_point_temperature_S', 'dew_point_temperature_W', 'holiday',
       'month', 'relative_humidity_C', 'relative_humidity_E',
       'relative_humidity_N', 'relative_humidity_S', 'relative_humidity_W',
       'temperature_C', 'temperature_E', 'temperature_N', 'temperature_S',
       'temperature_W', 'visibility_C', 'visibility_E', 'visibility_N',
  

In [261]:
er.columns

Index(['wind_gen_total_mw', 'wind_gen_west_mw', 'wind_gen_north_mw',
       'wind_gen_south_mw', 'wind_gen_coast_mw', 'wind_gen_panhandle_mw',
       'solar_gen_total_mw', 'solar_gen_west_mw', 'solar_gen_north_mw',
       'solar_gen_east_mw', 'solar_gen_south_mw', 'solar_gen_coast_mw',
       'COAST_Load', 'EAST_Load', 'ERCOT_Load', 'LZ_HOUSTON_DAM',
       'LZ_NORTH_DAM', 'LZ_RAYBN_DAM', 'LZ_SOUTH_DAM', 'LZ_WEST_DAM',
       'NORTH_Load', 'SOUTH_Load', 'WEST_Load', 'altimeter_C', 'altimeter_E',
       'altimeter_N', 'altimeter_S', 'altimeter_W', 'dew_point_temperature_C',
       'dew_point_temperature_E', 'dew_point_temperature_N',
       'dew_point_temperature_S', 'dew_point_temperature_W', 'holiday',
       'month', 'relative_humidity_C', 'relative_humidity_E',
       'relative_humidity_N', 'relative_humidity_S', 'relative_humidity_W',
       'temperature_C', 'temperature_E', 'temperature_N', 'temperature_S',
       'temperature_W', 'visibility_C', 'visibility_E', 'visibility_N',
  

In [254]:
regional_data = create_regional_dfs(er)

# Access individual regions
df_coast = regional_data['coast']
df_south = regional_data['south']
df_west  = regional_data['west']
df_north = regional_data['north']
df_east  = regional_data['east']

In [257]:
regional_data

{'coast':                       COAST_Load  LZ_HOUSTON_DAM  altimeter_C  \
 2022-06-28 21:00:00  17359.07147           98.83       1020.3   
 2022-06-28 22:00:00  16717.67041           79.97       1019.6   
 2022-06-28 23:00:00  15700.59693           64.97       1019.6   
 2022-06-29 00:00:00  14675.34967           56.02       1019.3   
 2022-06-29 01:00:00  13882.02086           53.30       1020.3   
 ...                          ...             ...          ...   
 2025-11-08 20:00:00          NaN           39.54          NaN   
 2025-11-08 21:00:00          NaN           32.77          NaN   
 2025-11-08 22:00:00          NaN           29.56          NaN   
 2025-11-08 23:00:00          NaN           27.85          NaN   
 2025-11-09 00:00:00          NaN           25.85          NaN   
 
                      dew_point_temperature_C  relative_humidity_C  \
 2022-06-28 21:00:00                     16.1                 34.0   
 2022-06-28 22:00:00                     19.0            

In [255]:
output_dir = 'regional_data'
os.makedirs(output_dir, exist_ok=True)

for region, df in regional_data.items():
    df.to_csv(f'{output_dir}/{region}.csv')
    print(f"✅ Saved: {output_dir}/{region}.csv")

✅ Saved: regional_data/coast.csv
✅ Saved: regional_data/south.csv
✅ Saved: regional_data/west.csv
✅ Saved: regional_data/north.csv
✅ Saved: regional_data/east.csv


In [258]:
# Export master df
er.to_csv(f'{output_dir}/er_master.csv')
print(f"✅ Saved: {output_dir}/er_master.csv")

✅ Saved: regional_data/er_master.csv


In [259]:
df_north

,NORTH_Load,LZ_NORTH_DAM,altimeter_N,dew_point_temperature_N,relative_humidity_N,temperature_N,visibility_N,holiday,weekend,month,Year
2022-06-28 21:00:00,1261.994240,97.10,1023.7,10.6,29.0,30.6,16.093,0,0,6,2022.0
2022-06-28 22:00:00,1224.577862,78.59,1023.0,10.6,29.0,30.6,16.093,0,0,6,2022.0
2022-06-28 23:00:00,1141.525414,64.00,1022.7,12.8,36.0,29.4,16.093,0,0,6,2022.0
2022-06-29 00:00:00,1050.896115,55.89,1022.0,12.8,35.0,30.0,16.093,0,0,6,2022.0
2022-06-29 01:00:00,995.840021,53.16,1022.0,12.8,35.0,30.0,16.093,0,0,6,2022.0
...,...,...,...,...,...,...,...,...,...,...,...
2025-11-08 20:00:00,NaN,28.76,NaN,NaN,NaN,NaN,NaN,0,1,11,NaN
2025-11-08 21:00:00,NaN,24.31,NaN,NaN,NaN,NaN,NaN,0,1,11,NaN
2025-11-08 22:00:00,NaN,21.71,NaN,NaN,NaN,NaN,NaN,0,1,11,NaN
2025-11-08 23:00:00,NaN,23.79,NaN,NaN,NaN,NaN,NaN,0,1,11,NaN
